# Unslothをつかったgptoss20bのSFT　→　失敗

CPT→SFTで、ハルシネーションの嵐が発生。よって、このノートブックは中断


### ライブラリロード

In [1]:
from unsloth import FastLanguageModel
import torch

# 128GBのメモリを活かし、最高精度（BF16）かつ大容量の文脈でロード
max_seq_length = 4096 # 200ページの情報を効率よくパッキングするために4096以上を推奨
dtype = torch.bfloat16 # DGX Spark/BlackwellならBF16が最適
load_in_4bit = False   # メモリに余裕があるため、量子化なしで精度を優先

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gpt-oss-20b", # Hugging Face上の正確なリポジトリ名を確認してください
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # device_map = "auto", # Unslothが自動でGPUに割り当てます
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.8: Fast Gpt_Oss patching. Transformers: 4.56.2.
   \\   /|    NVIDIA GB10. Num GPUs = 1. Max memory: 121.689 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0a0+b558c986e8.nv25.11. CUDA: 12.1. CUDA Toolkit: 13.0. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33+aa7bc36.d20260322. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Loading checkpoint shards: 100%|████████████████████████████████████████████████████████████████| 3/3 [01:28<00:00, 29.38s/it]


### PEFT/LoRAの設定

In [2]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 32, 
    # 特定の層だけでなく、すべての線形層（全結合層）を対象にします
    target_modules = "all-linear", 
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
)

Unsloth: Detected MoE model with num_experts = 32 and target_modules = '(?:.*?(?:vision|image|visual|patch|language|text).*?(?:self_attn|attention|attn|mlp|feed_forward|ffn|dense).*?(?:q_proj|k_proj|v_proj|o_proj).*?)|(?:\\bmodel\\.layers\\.[\\d]{1,}\\.(?:self_attn|attention|attn|mlp|feed_forward|ffn|dense)\\.(?:(?:q_proj|k_proj|v_proj|o_proj)))'. Enabling LoRA on MoE parameters: ['mlp.experts.gate_up_proj', 'mlp.experts.down_proj']
Unsloth: PEFT set target_parameters but found no matching parameters.
This is expected for MoE models - Unsloth handles MoE expert LoRA targeting separately.
Unsloth: Making `model.base_model.model.model` require gradients


ステップ2：作成したJSONLデータの読み込み

In [3]:
from datasets import load_dataset

# ファイルパスを指定して読み込み
dataset = load_dataset("json", data_files={"train": "cpt_input.jsonl"}, split="train")

# データの形式を確認（"text" カラムがあることを想定）
print(dataset[0])

{'text': '# 障害程度別対象事業一覧表 (○対象・△一部対象)\n\n以下は、障害程度別の対象となる事業の一覧です。\n\n- **東京メトロ旅客運賃の割引**（本文頁：120）視覚障害等級 1 から 6、聴覚又は平衡感覚機能障害等級 2 から 4 のすべてが対象となります。所得制限はありません。\n- **私鉄旅客運賃の割引**（本文頁：120）視覚障害等級 1 から 6、聴覚又は平衡感覚機能障害等級 2 から 4 のすべてが対象となります。所得制限はありません。\n- **タクシー運賃の割引**（本文頁：120）視覚障害等級 1 から 6、聴覚又は平衡感覚機能障害等級 2 から 4 のすべてが対象となります。所得制限はありません。\n- **航空運賃の割引**（本文頁：121）視覚障害等級 1 から 6、聴覚又は平衡感覚機能障害等級 2 から 4 のすべてが対象となります。所得制限はありません。\n- **旅客船運賃の割引**（本文頁：121）視覚障害等級 1 から 6、聴覚又は平衡感覚機能障害等級 2 から 4 のすべてが対象となります。所得制限はありません。\n- **テレビ受信料の減免**（本文頁：121）視覚障害等級 1 から 6、聴覚又は平衡感覚機能障害等級 2 から 4 のすべてが一部対象となります。所得制限があります。\n- **水道・下水道料金の減免**（本文頁：122）詳細は本文を参照してください。所得制限があります。\n- **携帯電話使用料等の割引**（本文頁：123）視覚障害等級 1 から 6、聴覚又は平衡感覚機能障害等級 2 から 4 のすべてが対象となります。所得制限はありません。\n- **郵便料金・ゆうパック運賃等の減免**（本文頁：123）詳細は本文を参照してください。所得制限はありません。\n- **通常郵便葉書（青い鳥郵便葉書）の無料配布**（本文頁：124）視覚障害等級 1、2、3 および聴覚又は平衡感覚機能障害等級 2 が対象となります。所得制限はありません。\n\n### **税の軽減**\n\n税の軽減に関する事業は以下の通りです。\n\n- **所得税・住民税の障害者控除**（本文頁：125）視覚障害等級 1 から 6、聴覚又は平衡感覚機能障害等級 2 から 4 のすべてが対象となります。所得制限はありま

### 学習

### 学習用メソッドの定義

In [4]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bf16_supported
import time

def train(args):
    trainer = SFTTrainer(
        model = model,
        tokenizer = tokenizer,
        train_dataset = dataset,
        dataset_text_field = "text",
        max_seq_length = 4096,
        packing = True, 
        args = args,
    )
    
    start_time = time.time()
    print(f"学習を開始します... (開始時刻: {time.ctime(start_time)})")
    # 学習開始！
    trainer_stats = trainer.train()
    # --- 学習終了後の時刻を記録 ---
    end_time = time.time()
    total_time = end_time - start_time
    
    # 秒を「分:秒」の形式に変換
    minutes = int(total_time // 60)
    seconds = int(total_time % 60)

    print("-" * 30)
    print(f"学習が完了しました！ (終了時刻: {time.ctime(end_time)})")
    print(f"合計学習時間: {minutes}分 {seconds}秒")
    print(f"1ステップあたりの平均時間: {total_time / trainer.args.max_steps:.2f}秒")
    print("-" * 30)
    

#### 学習用パラメータ

In [5]:
args = TrainingArguments(
        per_device_train_batch_size = 8, 
        gradient_accumulation_steps = 4, 
        warmup_steps = 50,
        max_steps = 200, 
        learning_rate = 4e-4, 
        lr_scheduler_type = "cosine", 
        bf16 = True, # DGX Spark/Blackwellなら必須の設定
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        output_dir = "outputs_gpt_oss_v3",
        save_steps = 100, # 途中で保存しておく
    )

#### 学習実行

In [7]:
train(args)

学習を開始します... (開始時刻: Sun Mar 22 23:22:13 2026)


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 218 | Num Epochs = 29 | Total steps = 200
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 4 x 1) = 32
 "-____-"     Trainable parameters = 15,925,248 of 1,820,384,832 (0.87% trained)
[unsloth_zoo.log|WARNING]Unsloth: GPT-OSS MoE training path: MXFP4 + triton_kernels. LoRA=False, experts=32. Tip: Increase batch_size for better GPU utilization.
Autotune Choices Stats:
{"num_choices": 23, "num_triton_choices": 23, "best_kernel": "triton_flex_attention_backward_7", "best_kernel_desc": "BLOCKS_ARE_CONTIGUOUS=False, BLOCK_M1=64, BLOCK_M2=64, BLOCK_N1=64, BLOCK_N2=64, FLOAT32_PRECISION=\"'tf32'\", GQA_SHARED_HEADS=8, HAS_FULL_BLOCKS=True, IS_DIVISIBLE=True, OUTPUT_LOGSUMEXP=True, OUTPUT_MAX=False, PRESCALE_QK=False, QK_HEAD_DIM=64, QK_HEAD_DIM_ROUNDED=64, ROWS_GUARANTEED_SAFE=False, SAFE_HEAD_DIM=True, SM_SCAL

Step,Training Loss
1,3.044800
2,3.046000
3,3.078000
4,3.046200
5,2.894800
6,2.722200
7,2.626000
8,2.429400
9,2.027700
10,1.784500


Autotune Choices Stats:
{"num_choices": 23, "num_triton_choices": 23, "best_kernel": "triton_flex_attention_backward_53", "best_kernel_desc": "BLOCKS_ARE_CONTIGUOUS=False, BLOCK_M1=64, BLOCK_M2=64, BLOCK_N1=64, BLOCK_N2=64, FLOAT32_PRECISION=\"'tf32'\", GQA_SHARED_HEADS=8, HAS_FULL_BLOCKS=True, IS_DIVISIBLE=True, OUTPUT_LOGSUMEXP=True, OUTPUT_MAX=False, PRESCALE_QK=False, QK_HEAD_DIM=64, QK_HEAD_DIM_ROUNDED=64, ROWS_GUARANTEED_SAFE=False, SAFE_HEAD_DIM=True, SM_SCALE=0.125, SPARSE_KV_BLOCK_SIZE=128, SPARSE_Q_BLOCK_SIZE=128, V_HEAD_DIM=64, V_HEAD_DIM_ROUNDED=64, WRITE_DQ=True, num_stages=3, num_warps=4", "best_time": 3.7541439533233643, "best_triton_pos": 0}
Autotune Choices Stats:
{"num_choices": 23, "num_triton_choices": 23, "best_kernel": "triton_flex_attention_backward_78", "best_kernel_desc": "BLOCKS_ARE_CONTIGUOUS=False, BLOCK_M1=64, BLOCK_M2=64, BLOCK_N1=64, BLOCK_N2=64, FLOAT32_PRECISION=\"'tf32'\", GQA_SHARED_HEADS=8, HAS_FULL_BLOCKS=True, IS_DIVISIBLE=True, OUTPUT_LOGSUMEXP=Tr

------------------------------
学習が完了しました！ (終了時刻: Mon Mar 23 00:38:51 2026)
合計学習時間: 76分 38秒
1ステップあたりの平均時間: 22.99秒
------------------------------


### 推論テスト

In [11]:
# プロンプトのテンプレートを定義
prompt_style = """以下は、障害者福祉の手続きに関するガイドラインに基づいた回答です。
質問に対して、正確な日本語で回答してください。

### 質問:
{}

### 回答:
"""

# 質問をテンプレートに埋め込む
input_text = prompt_style.format("身体障害者手帳の申請に必要なものと、電車を利用する際の手続きについて教えてください。")

inputs = tokenizer([input_text], return_tensors = "pt").to("cuda")

# 生成（パラメータを微調整：より確実な言葉を選ばせる）
outputs = model.generate(
    **inputs, 
    max_new_tokens = 300,
    temperature = 0.1,          # 低くして「迷い」を消す
    repetition_penalty = 1.2,   # 同じ言葉のループを防ぐ
    top_p = 0.9,
    use_cache = True
)

print(tokenizer.batch_decode(outputs, skip_special_tokens = True)[0])

以下は、障害者福祉の手続きに関するガイドラインに基づいた回答です。
質問に対して、正確な日本語で回答してください。

### 質問:
身体障害者手帳の申請に必要なものと、電車を利用する際の手続きについて教えてください。

### 回答:
東京メトロ旅客運賃の割引（120 ページ）は所得制限なく設用されておい。視覚障害 1級から6 級まで持つ方があり、それ自動障害 2 級から4 級まる等級も有べます。この制度では、すべても税金の軽減や郵便料金・ゆうパック運賄などの削除の対象となります。また、水道・下水道料金の減免については詳細な条件が本文表中記されていますので、具体的なおよび詈別貯�ーションの利上子助成には同様に全所無罪です。

携帯電話使用料等シんみたし工業賃の割引（123 ページ）も所得制限なしに適用されています。同様に、通常住宅設備改善費の非課税区隔があります。ただし、この場合「都営船」として参照していますため、視覚障害者手帳の種類によって異なる程度の支援内容がありば存在します。

## 税の軽減
個住民事財源の減額については、所得制限はありません。テレビ受信料の減免措置は所得制限なく、視覚障害 1-3 級、および自動障害 2 級


# モデルの保存

In [13]:
model.save_pretrained("gpt_oss_disabled_support_v1")
tokenizer.save_pretrained("gpt_oss_disabled_support_v1")


('gpt_oss_disabled_support_v1/tokenizer_config.json',
 'gpt_oss_disabled_support_v1/special_tokens_map.json',
 'gpt_oss_disabled_support_v1/chat_template.jinja',
 'gpt_oss_disabled_support_v1/tokenizer.json')

# 仕上げのSFT

In [14]:
import json
from datasets import Dataset
from trl import SFTTrainer
from transformers import TrainingArguments
import torch

# --- 1. sft_data.jsonl の読み込みとパース ---
data_list = []
with open("sft_data.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        data_list.append(json.loads(line))

# --- 2. プロンプトの型（フォーマット）を定義 ---
# 前回の推論で使用したテンプレートと一致させることが「正解」への近道です
def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    outputs      = examples["output"]
    texts = []
    for instruction, output in zip(instructions, outputs):
        # 知識(CPT)を活かしつつ、回答の形(SFT)を覚えさせる
        text = f"### 質問:\n{instruction}\n\n### 回答:\n{output}"
        texts.append(text)
    return { "text" : texts, }

# Datasetオブジェクトへの変換
dataset = Dataset.from_list(data_list)
dataset = dataset.map(formatting_prompts_func, batched = True)

# --- 3. 仕上げの学習（SFT）設定 ---
# 知識(CPT)は既にあるので、低い学習率で「喋り方」だけを矯正します
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = 2048,
    packing = False, # SFTではパッキングなしの方が精度が安定します
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,            # データの量によりますが、60回も回せば十分です
        learning_rate = 5e-5,      # CPT(4e-4)より大幅に下げ、知識を壊さないようにします
        bf16 = True,               # DGX Sparkなら必須
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs_sft_v1",
    ),
)

# 特訓開始！
trainer.train()

Map: 100%|████████████████| 4/4 [00:00<00:00, 676.42 examples/s]
num_proc must be <= 4. Reducing num_proc to 4 for dataset of size 4.
[datasets.arrow_dataset|WARNING]num_proc must be <= 4. Reducing num_proc to 4 for dataset of size 4.
Unsloth: Tokenizing ["text"] (num_proc=4): 100%|█| 4/4 [00:02<00
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 4 | Num Epochs = 60 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 15,925,248 of 1,820,384,832 (0.87% trained)
Autotune Choices Stats:
{"num_choices": 23, "num_triton_choices": 23, "best_kernel": "triton_flex_attention_backward_110", "best_kernel_desc": "BLOCKS_ARE_CONTIGUOUS=False, BLOCK_M1=32, BLOCK_M2=32, BLOCK_N1=32, BLOCK_N2=32, FLOAT32_PRECISION=\"'tf32'\", GQA_SHARED_HEADS=8, HAS_FULL_BLOCKS=True, IS_DIVISIBLE=False, OUTPUT_LOGSUMEXP=T

Step,Training Loss
1,5.072800
2,5.072800
3,5.021000
4,4.818100
5,4.683500
6,4.429200
7,4.234000
8,4.017400
9,3.831200
10,3.659700


TrainOutput(global_step=60, training_loss=2.5263566354910534, metrics={'train_runtime': 48.8167, 'train_samples_per_second': 9.833, 'train_steps_per_second': 1.229, 'total_flos': 195943944741120.0, 'train_loss': 2.5263566354910534, 'epoch': 60.0})

In [16]:
FastLanguageModel.for_inference(model)

prompt_style = """### 質問:
{}

### 回答:有料道路の割引を受けるには、
"""

# 質問：先ほど崩れていた「有料道路」や「鉄道」について
question = "有料道路の割引を受けるための申請手続きと、必要なものを教えてください。"

inputs = tokenizer([prompt_style.format(question)], return_tensors = "pt").to("cuda")

outputs = model.generate(
    **inputs, 
    max_new_tokens = 300,
    temperature = 0.1,    # 低く設定して、学習した「型」を忠実に守らせる
    repetition_penalty = 1.2,
    use_cache = True
)

print(tokenizer.batch_decode(outputs, skip_special_tokens = True)[0])

### 質問:
有料道路の割引を受けるための申請手続きと、必要なものを教えてください。

### 回答:有料道路の割引を受けるには、
東京メトロ運賃や私鉄タクシー等の運賁減害において、所得制限はなくありません。身体障害者又持ち旅客・聽覚機能障害が無保直です。
視覚�平感不動乗車（かどん）の税上非課免除があります。
関税的豴置成についても同様に全て不可条件であるので、すべず本文ページ120ではあらゆよう要件が設付されています。

通�た船便 (126)
テレビ道信所の税控 adj得制度は不可条件としているま、所得制限は存在します。身体障害者又持ち旅客設備傷害者が無頶料金となっています。
**テン使用事 telephone費
既婚船运購営の税上優待制度はましてい！この制度対象外ですが、詳細は本文頁121をご参照ください。
*高額耐 endurance handicap：134

## 税前の軽くうよサ,parent貯人子方�公子などの税緘ORYの非課金措置は所得制限ありございます。この制度適用する交通記録は以下に示しています：

-Xせキエポートの税上優転制度は123ページに記載されてありますが、このページでは一部兼容性


### うまくいかないので、追加のSFT

In [17]:
# 1. sft_data.jsonl の読み込み
data_list = []
try:
    with open("sft_data.jsonl", "r", encoding="utf-8") as f:
        for line in f:
            data_list.append(json.loads(line))
    print(f"成功: {len(data_list)} 件のデータを読み込みました。")
except FileNotFoundError:
    print("エラー: sft_data.jsonl が見つかりません。ファイルを作成してから再実行してください。")

# 2. プロンプトフォーマットの定義
def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    outputs      = examples["output"]
    texts = []
    for instruction, output in zip(instructions, outputs):
        # 推論時と同じ型を学習させることで、迷いをなくします
        text = f"### 質問:\n{instruction}\n\n### 回答:\n{output}"
        texts.append(text)
    return { "text" : texts, }

dataset = Dataset.from_list(data_list)
dataset = dataset.map(formatting_prompts_func, batched = True)

# 3. 日本語回路「修復」のための学習設定
# 学習率を 1e-5 (極低速) にし、ステップ数を増やして反復横跳びのように覚えさせます
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = 2048,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        max_steps = 300,            # 30件×10周ほど回して、日本語の型を強制固定
        learning_rate = 1e-5,       # 非常に低くし、今の知識(Loss 0.37)を壊さない
        bf16 = True,
        logging_steps = 5,
        optim = "adamw_8bit",
        lr_scheduler_type = "cosine",
        seed = 3407,
        output_dir = "outputs_sft_rescue",
        report_to = "none",         # ログをシンプルに
    ),
)

# 特訓開始（DGX Sparkなら3〜5分で完了します）
trainer.train()

成功: 34 件のデータを読み込みました。


Map: 100%|████████████| 34/34 [00:00<00:00, 10286.83 examples/s]
Unsloth: Tokenizing ["text"] (num_proc=24): 100%|█| 34/34 [00:07
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 34 | Num Epochs = 60 | Total steps = 300
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 15,925,248 of 1,820,384,832 (0.87% trained)


Step,Training Loss
5,3.729600
10,3.394100
15,3.585200
20,3.359400
25,3.321300
30,3.276900
35,3.078100
40,2.976200
45,2.959000
50,3.070600


TrainOutput(global_step=300, training_loss=2.773389075597127, metrics={'train_runtime': 172.4104, 'train_samples_per_second': 13.92, 'train_steps_per_second': 1.74, 'total_flos': 1393473497709312.0, 'train_loss': 2.773389075597127, 'epoch': 60.0})

In [18]:
FastLanguageModel.for_inference(model)

# 質問
question = "有料道路の割引を受けるための申請手続きと、必要なものを教えてください。"
# 以前のように回答を無理やり書き込まず、まずはモデルに任せてみます
prompt = f"### 質問:\n{question}\n\n### 回答:\n"

inputs = tokenizer([prompt], return_tensors = "pt").to("cuda")

outputs = model.generate(
    **inputs, 
    max_new_tokens = 300,
    num_beams = 5,              # 5つの候補を比較して最高のものを選ぶ
    no_repeat_ngram_size = 3,   # 「運賃運賃」のようなループを防止
    repetition_penalty = 1.5,   # 変な造語への執着を捨てる
    early_stopping = True,
    use_cache = True
)

print(tokenizer.batch_decode(outputs, skip_special_tokens = True)[0])

### 質問:
有料道路の割引を受けるための申請手続きと、必要なものを教えてください。

### 回答:
東京メトロ運賃や私鉄、タクシー、航空運�ック、旅客運船の減免制度について、障害者等に対して 100%の規定が設置されました。これらの制度は所得制限がありませんので、すべての方が対象となります。

### **税の軽減:
所得税、人事税、住民税、相続税、少子贈
個人事業業税、公共料金の非課税制度についても一部別割引が適用されます。詳細は以下の通りです。
### **###回答詳細について:
税の航空業務員控金,住民登録の非登録事務金,少額貯ドの利子、あらは所得税や相形税の免除、自動車の割割成関ック事の免れ、特定保護設備（高いサポート設備）
詳細は本文参照してください。

######Jタク業金: 
JV工業船業金、住宅の改善費子事業金など 120%の記載があり、同様に一部対象となる規定を適用します。詳細をご参照ください。

####**###Excelデータ修正

###ペタ房
###△修築・改修事金
###〇△


### CPT後の状態から再度SFTを実行

In [2]:
import json
import torch
from datasets import Dataset
from trl import SFTTrainer
from transformers import TrainingArguments

# 1. sft_data.jsonl の読み込み
data_list = []
try:
    with open("sft_data.jsonl", "r", encoding="utf-8") as f:
        for line in f:
            data_list.append(json.loads(line))
    print(f"成功: {len(data_list)} 件のデータを読み込みました。")
except FileNotFoundError:
    print("エラー: sft_data.jsonl が見つかりません。ファイルを作成してから再実行してください。")

# 2. プロンプトフォーマットの定義
def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    outputs      = examples["output"]
    texts = []
    for instruction, output in zip(instructions, outputs):
        # 推論時と同じ型を学習させることで、迷いをなくします
        text = f"### 質問:\n{instruction}\n\n### 回答:\n{output}"
        texts.append(text)
    return { "text" : texts, }

dataset = Dataset.from_list(data_list)
dataset = dataset.map(formatting_prompts_func, batched = True)

成功: 34 件のデータを読み込みました。


Map: 100%|██████████████████████████████████████████████████████████████████████████| 34/34 [00:00<00:00, 19410.15 examples/s]


In [3]:
# 1. まずベースモデル（本体）をロードする
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gpt-oss-20b-bnb-4bit", # 元のベースモデル名
    max_seq_length = 2048,
    load_in_4bit = True,
)

# 2. 保存しておいた LoRAアダプタ（差分）をロードして合体させる
model = FastLanguageModel.get_peft_model(
    model,
    model_name = "gpt_oss_disabled_support_v1", # 保存したフォルダ名
)

print("モデルの復元に成功しました。これからリベンジ学習を開始します。")

# 2. SFTの設定（日本語を壊さない安全な設定）
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset, # 先ほどの30件データ
    dataset_text_field = "text",
    max_seq_length = 2048,
    args = TrainingArguments(
        per_device_train_batch_size = 4, # DGXならもっと上げてもOK
        gradient_accumulation_steps = 4,
        warmup_steps = 20,
        max_steps = 500,           # じっくり500ステップ
        learning_rate = 2e-5,      # ★ここが重要！絶対に日本語を壊さない低学習率
        bf16 = True,
        logging_steps = 1,
        optim = "adamw_8bit",
        lr_scheduler_type = "cosine",
        output_dir = "outputs_final_rescue",
    ),
)
trainer.train()

==((====))==  Unsloth 2026.3.8: Fast Gpt_Oss patching. Transformers: 4.56.2.
   \\   /|    NVIDIA GB10. Num GPUs = 1. Max memory: 121.689 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0a0+b558c986e8.nv25.11. CUDA: 12.1. CUDA Toolkit: 13.0. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33+aa7bc36.d20260322. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards: 100%|████████████████████████████████████████████████████████████████| 9/9 [04:23<00:00, 29.28s/it]


Unsloth: Detected MoE model with num_experts = 32 and target_modules = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']. Enabling LoRA on MoE parameters: ['mlp.experts.gate_up_proj', 'mlp.experts.down_proj']


/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:212: UserWarning: Unsupported layer type '<class 'transformers.models.gpt_oss.modeling_gpt_oss.GptOssExperts'>' encountered, proceed at your own risk.
  warnings.warn(f"Unsupported layer type '{type(module)}' encountered, proceed at your own risk.", UserWarning)


Unsloth: Making `model.base_model.model.model` require gradients
モデルの復元に成功しました。これからリベンジ学習を開始します。


Unsloth: Tokenizing ["text"] (num_proc=24): 100%|██████████████████████████████████████| 34/34 [00:08<00:00,  4.00 examples/s]
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 34 | Num Epochs = 167 | Total steps = 500
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 184,909,824 of 21,099,667,008 (0.88% trained)
Autotune Choices Stats:
{"num_choices": 23, "num_triton_choices": 23, "best_kernel": "triton_flex_attention_backward_15", "best_kernel_desc": "BLOCKS_ARE_CONTIGUOUS=False, BLOCK_M1=32, BLOCK_M2=32, BLOCK_N1=32, BLOCK_N2=32, FLOAT32_PRECISION=\"'tf32'\", GQA_SHARED_HEADS=8, HAS_FULL_BLOCKS=True, IS_DIVISIBLE=False, OUTPUT_LOGSUMEXP=True, OUTPUT_MAX=False, PRESCALE_QK=False, QK_HEAD_DIM=64, QK_HEAD_DIM_ROUNDED=64, ROWS_GUARANTEED_SAFE=False, SAFE_HEAD_DIM=True, SM_SCALE=0.125, SPARSE_KV_BLOCK_SIZE=1

Step,Training Loss
1,2.086600
2,2.082900
3,2.019900
4,2.080200
5,2.030800
6,2.502700
7,2.153400
8,2.022300
9,1.965700
10,2.123600


Autotune Choices Stats:
{"num_choices": 23, "num_triton_choices": 23, "best_kernel": "triton_flex_attention_backward_109", "best_kernel_desc": "BLOCKS_ARE_CONTIGUOUS=False, BLOCK_M1=32, BLOCK_M2=32, BLOCK_N1=32, BLOCK_N2=32, FLOAT32_PRECISION=\"'tf32'\", GQA_SHARED_HEADS=8, HAS_FULL_BLOCKS=True, IS_DIVISIBLE=False, OUTPUT_LOGSUMEXP=True, OUTPUT_MAX=False, PRESCALE_QK=False, QK_HEAD_DIM=64, QK_HEAD_DIM_ROUNDED=64, ROWS_GUARANTEED_SAFE=False, SAFE_HEAD_DIM=True, SM_SCALE=0.125, SPARSE_KV_BLOCK_SIZE=128, SPARSE_Q_BLOCK_SIZE=128, V_HEAD_DIM=64, V_HEAD_DIM_ROUNDED=64, WRITE_DQ=True, num_stages=2, num_warps=4", "best_time": 0.11772800236940384, "best_triton_pos": 0}
Autotune Choices Stats:
{"num_choices": 23, "num_triton_choices": 23, "best_kernel": "triton_flex_attention_backward_134", "best_kernel_desc": "BLOCKS_ARE_CONTIGUOUS=False, BLOCK_M1=32, BLOCK_M2=32, BLOCK_N1=32, BLOCK_N2=32, FLOAT32_PRECISION=\"'tf32'\", GQA_SHARED_HEADS=8, HAS_FULL_BLOCKS=True, IS_DIVISIBLE=False, OUTPUT_LOGSUME

TrainOutput(global_step=500, training_loss=0.6110218678414822, metrics={'train_runtime': 1470.284, 'train_samples_per_second': 5.441, 'train_steps_per_second': 0.34, 'total_flos': 6.968379205448294e+16, 'train_loss': 0.6110218678414822, 'epoch': 166.88888888888889})

In [4]:
from unsloth import FastLanguageModel
FastLanguageModel.for_inference(model)

# 質問
question = "身体障害者手帳を使って、有料道路の割引を受けるための具体的な手続きと必要なものを教えてください。"
prompt = f"### 質問:\n{question}\n\n### 回答:\n"

inputs = tokenizer([prompt], return_tensors = "pt").to("cuda")

# 丁寧に生成
outputs = model.generate(
    **inputs, 
    max_new_tokens = 300,
    temperature = 0.1,    # 学習した「正解」を忠実に出力させる
    repetition_penalty = 1.2,
    use_cache = True
)

print(tokenizer.batch_decode(outputs, skip_special_tokens = True)[0])

### 質問:
身体障害者手帳を使って、有料道路の割引を受けるための具体的な手続きと必要なものを教えてください。

### 回答:
1. **交番での申請**: まず、各種公共料金の減額を認める「介護者用保健福祉インターネット発行ハンコ」（通称“ひんこ”）を取得するために、お住まいの地域の警察署または交番へ入ります。ここで、「本人確認書類」とともに「介護者用ヒンク」を提示し、申請します。
   
2. **区役所窓口での登録**: 次に、その“ひんこ”（切符形状の紙）を持ち込み、お勤労者級別などのお住まみの区役所や市役所の窓口で、身体障害者手帳との併用による交通費の減額を申し込むことができます。この際、以下の資料が必要です：
   - 身体障害者手帳（原本）
   - “ひんこ”
   - 本人確認書類（運転免許証・パスポート等）

3. **高速道路の割引**: 上記の手続きを済ませた後、高速道路の券売機で“ひんこ”を提示すると、通常の運賃から減額されます。また、市区町村管轄のトールボックスでも同様に提示可能
